# Navigator context MCP — demo

This notebook wires the **navigator_context** MCP server into the OpenAI Agents SDK. The server keeps a **problem statement**, a **success metric**, and **session notes**, then can emit a **briefing** string you can fold into prompts—useful when you want explicit, inspectable context instead of hiding everything inside model memory.

**Requirements:** `OPENAI_API_KEY` in your environment (e.g. `.env` at the repo root). Run `uv sync` at the repo root if needed.

**Optional:** Set `OPENAI_API_BASE` (or `OPENAI_BASE_URL`) to use an OpenAI-compatible gateway—e.g. OpenRouter (`https://openrouter.ai/api/v1`). When set, the notebook configures the Agents SDK with `set_default_openai_client`. For OpenRouter-style gateways, set **`OPENAI_MODEL`** to a provider-qualified name (e.g. `openai/gpt-4o-mini`); the agent cell uses that env var when present.

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, set_default_openai_api, set_default_openai_client
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display

load_dotenv(override=True)

# Optional: non-default host (OpenRouter, Azure, local OpenAI-compatible server, etc.)
_api_base = (os.getenv("OPENAI_API_BASE") or os.getenv("OPENAI_BASE_URL") or "").strip()
_api_key = os.getenv("OPENAI_API_KEY", "").strip()
if _api_base:
    if not _api_key:
        raise ValueError("OPENAI_API_BASE / OPENAI_BASE_URL is set but OPENAI_API_KEY is empty.")
    set_default_openai_api("chat_completions")
    set_default_openai_client(
        AsyncOpenAI(base_url=_api_base, api_key=_api_key),
        use_for_tracing=False,
    )
    print("Using OpenAI-compatible API base:", _api_base)


def find_server_dir() -> Path:
    here = Path.cwd().resolve()
    if (here / "navigator_server.py").exists():
        return here
    rel = here / "6_mcp/community_contributions/erisanolasheni/navigator_context_mcp"
    if rel.is_dir() and (rel / "navigator_server.py").exists():
        return rel
    raise FileNotFoundError(
        "Could not find navigator_server.py. Run Jupyter from the repo root or from this folder."
    )


SERVER_DIR = find_server_dir()
print("MCP cwd:", SERVER_DIR)

MCP cwd: /Users/hakeemerisan/development/codes/andela/ai-engineering/agents/6_mcp/community_contributions/erisanolasheni/navigator_context_mcp


In [6]:
params = {
    "command": "uv",
    "args": ["run", "navigator_server.py"],
    "cwd": str(SERVER_DIR),
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()
    print([t.name for t in tools])

['set_problem_statement', 'set_success_metric', 'append_session_note', 'list_session_notes', 'get_briefing_for_prompt', 'clear_navigator_state']


## Agent that uses the navigator tools

The agent can set the problem and metric, log what it tried, and fetch the briefing. Keep instructions small and trace the run in the SDK UI.

In [7]:
instructions = """You help a builder clarify an agentic project using the navigator tools.

1. If the user describes a vague goal, ask one short clarifying question OR infer a crisp problem and record it with set_problem_statement.
2. Propose one concrete success metric and save it with set_success_metric.
3. Log what you did with append_session_note.
4. End by calling get_briefing_for_prompt and quoting that briefing in your final reply so the user sees the stored context.
"""

model = os.getenv("OPENAI_MODEL") or "gpt-4o-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp:
    agent = Agent(
        name="Navigator assistant",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp],
    )
    with trace("Navigator MCP demo"):
        result = await Runner.run(
            agent,
            "I want something that helps our team run better standups without another heavy app.",
        )

display(Markdown(result.final_output))

Error getting response: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-or-v1*************************************************************639a. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}. (request_id: req_b149c7a829864fe7b1660cee405cc79a)


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-or-v1*************************************************************639a. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}